# Cross-encoder, round 2 (Business Entity Resolution)

Continues fine-tuning the round-1 model (`MyDrive/er_ce/model`, from `cross_encoder.ipynb`)
on hard pairs of train entities that neither the pair model nor round 1 used
(`ce2_train.parquet`, which also holds confident pairs of the countries without
training data, labeled by the pair model itself), then rescores every hard pair of the
current pair model (`ce2_oof.parquet`, `ce2_test.parquet`, and `ce2_oof_loco.parquet` if
present). Same model family: `FacebookAI/xlm-roberta-base` (MIT, 278M parameters), only
the provided challenge data.

* **Run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.
* **Training** (~60 min) needs `ce2_train.parquet` in `MyDrive/er_ce/`. If `ce2_oof` /
  `ce2_test` are not uploaded yet, the notebook stops after training; upload them later
  and **Run all** again (training is then skipped).
* **Output:** `MyDrive/er_ce/ce2_scores_oof.parquet`, `ce2_scores_test.parquet` (scored
  first), then `ce2_scores_oof_loco.parquet`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
DIR = '/content/drive/MyDrive/er_ce'
assert os.path.exists(f'{DIR}/model/config.json'), 'round-1 model missing: MyDrive/er_ce/model'
assert os.path.exists(f'{DIR}/ce2_train.parquet'), 'upload ce2_train.parquet to MyDrive/er_ce first'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

In [ ]:
import math, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, log_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
MODEL_DIR1 = f'{DIR}/model'    # round 1
MODEL_DIR2 = f'{DIR}/model2'   # round 2 (written here)
MAX_LEN = 96
BATCH, EVAL_BATCH = 32, 256
LR, EPOCHS, WARMUP = 1e-5, 1, 0.03   # lower learning rate: continuing from round 1
SEED = 42
DEV = 'cuda'
torch.manual_seed(SEED)
np.random.seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_DIR1)


def batches(df, bs, shuffle=False, seed=SEED):
    idx = np.arange(len(df))
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    for s in range(0, len(idx), bs):
        b = df.iloc[idx[s:s + bs]]
        enc = tok(list(b.text_a), list(b.text_b), truncation='longest_first',
                  max_length=MAX_LEN, padding=True, return_tensors='pt')
        yield b, {k: v.to(DEV) for k, v in enc.items()}


@torch.no_grad()
def logits_of(model, df):
    model.eval()
    out = []
    with torch.autocast('cuda', dtype=torch.float16):
        for _, enc in batches(df, EVAL_BATCH):
            out.append(model(**enc).logits.squeeze(-1).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)


def report(name, val, score):
    hard = val.p.between(0.02, 0.98).values
    for part, m in (('all held-out pairs', np.ones(len(val), bool)), ('hard pairs only', hard)):
        yv = val.label.values[m]
        print(f'{name} | {part}: {m.sum()} pairs | AUC {roc_auc_score(yv, score[m]):.4f} | '
              f'log loss {log_loss(yv, np.clip(score[m], 1e-6, 1 - 1e-6)):.4f}')

## 1. Continue fine-tuning (skipped if `model2` is already on Drive)

In [ ]:
if os.path.exists(f'{MODEL_DIR2}/config.json'):
    print('round-2 model found on Drive, skipping training')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR2).to(DEV)
else:
    data = pd.read_parquet(f'{DIR}/ce2_train.parquet')
    held = pd.util.hash_pandas_object(data.s1_id, index=False).values % 50 == 0
    val, tr = data[held], data[~held]
    print(f'train {len(tr)} pairs, validation {len(val)}, positive rate {tr.label.mean():.3f}')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR1).to(DEV)
    report('pair model', val, val.p.values)
    report('round 1   ', val, 1 / (1 + np.exp(-logits_of(model, val))))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(tr) / BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler('cuda')
    loss_fn = torch.nn.BCEWithLogitsLoss()
    step, t0 = 0, time.time()
    for ep in range(EPOCHS):
        model.train()
        for b, enc in batches(tr, BATCH, shuffle=True, seed=SEED + 10 + ep):
            y = torch.tensor(b.label.values, dtype=torch.float32, device=DEV)
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(**enc).logits.squeeze(-1)
            loss = loss_fn(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            if step % 500 == 0:
                print(f'step {step}/{steps}  loss {loss.item():.4f}  {time.time() - t0:.0f}s')
    model.save_pretrained(MODEL_DIR2)
    tok.save_pretrained(MODEL_DIR2)
    report('round 2   ', val, 1 / (1 + np.exp(-logits_of(model, val))))

## 2. Rescore every hard pair of the current pair model (resumable)

In [ ]:
def score(name, chunk=100_000):
    out = f'{DIR}/ce2_scores_{name}.parquet'
    if os.path.exists(out):
        print('already done:', out)
        return
    df = pd.read_parquet(f'{DIR}/ce2_{name}.parquet', columns=['s1_id', 'match_id', 'text_a', 'text_b'])
    order = np.argsort((df.text_a.str.len() + df.text_b.str.len()).values, kind='stable')
    df = df.iloc[order].reset_index(drop=True)
    parts, t0 = [], time.time()
    for c0 in range(0, len(df), chunk):
        part = f'{DIR}/tmp2_{name}_{c0}.parquet'
        if not os.path.exists(part):
            sub = df.iloc[c0:c0 + chunk]
            pd.DataFrame({'s1_id': sub.s1_id.values, 'match_id': sub.match_id.values,
                          'ce_logit': logits_of(model, sub)}).to_parquet(part)
        parts.append(pd.read_parquet(part))
        print(f'{name}: {min(c0 + chunk, len(df))}/{len(df)}  {time.time() - t0:.0f}s')
    pd.concat(parts, ignore_index=True).to_parquet(out)
    for c0 in range(0, len(df), chunk):
        os.remove(f'{DIR}/tmp2_{name}_{c0}.parquet')
    print('wrote', out, '-> ready to download')


# oof and test first (they feed the next upload), then any extra file such as oof_loco
names = sorted((f[4:-8] for f in os.listdir(DIR)
                if f.startswith('ce2_') and f.endswith('.parquet')
                and not f.startswith('ce2_scores_') and f != 'ce2_train.parquet'),
               key=lambda n: (n not in ('oof', 'test'), n))
if not names:
    print('Training done. Upload ce2_oof.parquet, ce2_test.parquet and ce2_oof_loco.parquet (src/ce_export.py --round2 score), '
          'then Runtime > Run all again.')
for name in names:
    score(name)
if names:
    print('Done. Download from MyDrive/er_ce/ into student_resource/work/ce/:',
          [f'ce2_scores_{n}.parquet' for n in names])